# Triple Pendulum Chaos Exploration

This notebook demonstrates the simulation pipeline: building an initial-condition
grid, running a batch RK4 simulation, inspecting flip-time statistics, and
visualizing 2D slices through the 3D chaos voxel map.

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on sys.path so `src` is importable
project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt

from src.utils.grid import make_grid
from src.simulation.batch_sim import simulate_batch
from src.utils.io import save_results_json, load_results_json
from src.visualization.colormap import get_matplotlib_cmap, flip_time_to_color

## Small Simulation

Run a 10x10x10 grid (1,000 initial conditions) with `t_max=10.0` seconds.
Each point starts from rest (all angular velocities zero) with a unique
combination of `(theta_1, theta_2, theta_3)` spanning +/-170 degrees.

In [ ]:
grid_size = 10
grid = make_grid(grid_size)
print(f"Grid shape: {grid.shape}  ({grid.shape[0]} initial conditions)")

results = simulate_batch(grid, dt=0.01, t_max=10.0)

## Inspect Results

The simulation returns a dictionary with `flip_times` (NaN if the pendulum
never flipped), `final_states`, and `metadata`. Below we compute basic
statistics on the flip-time distribution.

In [ ]:
flip_times = results["flip_times"]
flipped_mask = np.isfinite(flip_times)
flipped_times = flip_times[flipped_mask]

total_points = flip_times.size
num_flipped = flipped_mask.sum()
percent_flipped = 100.0 * num_flipped / total_points

print(f"Total points:  {total_points}")
print(f"Flipped:       {num_flipped} ({percent_flipped:.1f}%)")
print(f"Never flipped: {total_points - num_flipped}")
if num_flipped > 0:
    print(f"Min flip time: {flipped_times.min():.4f} s")
    print(f"Max flip time: {flipped_times.max():.4f} s")
    print(f"Mean flip time: {flipped_times.mean():.4f} s")
else:
    print("No pendulums flipped within the simulation window.")

## 2D Slice Visualization

Reshape the flat flip-time array into a `(n, n, n)` volume, then extract
a single 2D slice at `theta_3 = 0` (the middle index of the grid).
The custom `chaos_magma` colormap maps fast flips to dark purple and slow
flips to white, with NaN (never flipped) shown in black.

In [ ]:
chaos_cmap = get_matplotlib_cmap("chaos_magma")
volume = flip_times.reshape(grid_size, grid_size, grid_size)

# Middle index corresponds to theta_3 ~ 0 degrees
middle_index = grid_size // 2
theta3_value = np.linspace(-170, 170, grid_size)[middle_index]
slice_2d = volume[:, :, middle_index]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(
    slice_2d.T,
    origin="lower",
    extent=[-170, 170, -170, 170],
    cmap=chaos_cmap,
    vmin=0,
    vmax=10.0,
    aspect="equal",
)
ax.set_xlabel(r"$\theta_1$ (degrees)")
ax.set_ylabel(r"$\theta_2$ (degrees)")
ax.set_title(f"Flip time at $\\theta_3 = {theta3_value:.0f}°$")
fig.colorbar(im, ax=ax, label="Time to first flip (s)")
plt.tight_layout()
plt.show()

## Multiple Slices

Show a 2x3 grid of slices at six evenly spaced `theta_3` values across the
full range. This reveals how the chaos structure evolves along the third axis.

In [ ]:
theta3_axis = np.linspace(-170, 170, grid_size)
slice_indices = np.linspace(0, grid_size - 1, 6, dtype=int)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, slice_index in zip(axes.flat, slice_indices):
    slice_data = volume[:, :, slice_index]
    im = ax.imshow(
        slice_data.T,
        origin="lower",
        extent=[-170, 170, -170, 170],
        cmap=chaos_cmap,
        vmin=0,
        vmax=10.0,
        aspect="equal",
    )
    ax.set_title(f"$\\theta_3 = {theta3_axis[slice_index]:.0f}°$")
    ax.set_xlabel(r"$\theta_1$")
    ax.set_ylabel(r"$\theta_2$")

fig.colorbar(im, ax=axes, label="Time to first flip (s)", shrink=0.8)
fig.suptitle("Triple Pendulum Flip Time — Slices along $\\theta_3$", fontsize=14)
plt.tight_layout()
plt.show()

## Colormap Demo

Display the `chaos_magma` colormap as a horizontal gradient bar, showing
the full progression from fast-flip (dark indigo) to slow-flip (white).

In [ ]:
gradient = np.linspace(0, 1, 256).reshape(1, -1)

fig, ax = plt.subplots(figsize=(10, 1.5))
ax.imshow(gradient, aspect="auto", cmap=chaos_cmap)
ax.set_xticks([0, 64, 128, 192, 255])
ax.set_xticklabels(["0.0", "0.25", "0.50", "0.75", "1.0"])
ax.set_yticks([])
ax.set_xlabel("Normalized flip time")
ax.set_title("chaos_magma colormap")
plt.tight_layout()
plt.show()